## Linearization Pipeline

This notebook has code that will run the entire linearization process on a single data file.

#### Instructions

1. Edit the assignment statements in the code cell below to define paths to data files and other options (eventually these will be read from the command line or a configuration file).

2. Execute all the code cells.  You can click "Run All" to execute every step, then scroll to the bottom to view the plot showing the linearized data set.  Alternatively, step through the code once cell at a time.  At various places where will be code cells that have been commented out; uncomment them if you want to see a more detailed output.

In [1]:
# Define the path to the project directory.  It can be an absolute path or a path relative to this notebook.

from pathlib import Path

TEST_DATA = '210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched.xlsx'

PROJECT_DIR = Path.home() / 'Research/Projects/LibudaLab/GAP/PRG-1'
IMARIS_FILE = PROJECT_DIR / 'xlsx' / TEST_DATA
MEIOTIC_STAGES = PROJECT_DIR / 'Germline Measurements.xlsx'

#### Notebook Organization

Each step in the analysis pipeline is in a separate section that starts with a level 3 headers ("Read Position Data", "Compute Perpendicular Intersections", _etc_.)

Most sections define one or more Python functions and then call those functions.  The idea is to simplify the notebook's global namespace.
* objects that only used temporarily as part of a single step are saved in local variables of the functions
* any data that will be used in later steps is saved in a global variable

Local variables typically have short names that should be understandable in context, _e.g._ `mf` stands for "measurement frame" in the section that reads the line segment locations.

Global variables will have longer names that should be recognizable later, _e.g._ `segments` will be a frame that has all the information about line segments.  To make it easier to find cells where the variables are defined comments before the assignment statement are marked with ★

### Imports

In [2]:
# Installed data science libraries 
import geopandas as gp
import numpy as np
import pandas as pd
from shapely.geometry import Point, LineString

# Installed graphics libraries
from bokeh.palettes import Sunset10, Bright3
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.transform import linear_cmap, factor_cmap


### Read Position Data

Define a function that gets $x$ and $y$ coordinates from the Position sheet of the Imaris data file.  The function will be called twice, once to read measurement (ends of line segments) locations and once to read locations of nuclei.

In [3]:
def read_positions(f: pd.ExcelFile, category: str):
    '''
    Read the "Position" sheet from an XLS file exported by Imaris.  Select data points
    in the specified category, return them in a data frame that has the coordinates and
    IDs of the selected data.  Raises an exception if the file does not have a sheet 
    named "Position".

    Arguments:
        f:  the file to read from
        category: the type of data to read (based on the Category column in the sheet)

    Returns:
        a Pandas frame with names and Point objects
    '''
    assert "Position" in f.sheet_names, "spreadheet is missing the 'Position' sheet"

    sheet = f.parse("Position", header=1)

    pf = sheet[sheet['Category']==category]
    pf.index = range(len(pf))

    if category == 'MeasurementPoint':
        id_col = pd.Series(pf['Name'])
        id_name = 'name'
    else:
        id_col = pd.Series((pf['Surpass Object'] + pf['ID'].apply(str)), name='ID')
        id_name = 'id'

    df = gp.GeoDataFrame({
        id_name: id_col,
        'point': [Point(pf.loc[i]['Position X'], pf.loc[i]['Position Y']) for i in range(len(pf))]
    }).set_geometry('point')

    return df


★ Read the end points of the line segments and the locations of nuclei, save them in global variables for future steps.

In [4]:
with pd.ExcelFile(IMARIS_FILE, engine='calamine') as f:
    measurements = read_positions(f, category='MeasurementPoint')
    nuclei = read_positions(f, category='Surface')

**Optional:**  Uncomment this code cell to see the first few measurements.

In [5]:
measurements.head()

,name,point
0,A,POINT (38.501 141.542)
1,B,POINT (39.457 120.56)
2,C,POINT (40.56 110.826)
3,D,POINT (43.907 99.405)
4,E,POINT (48.822 81.404)


**Optional:**  Uncomment this code cell to see the first few nuclei.

In [6]:
nuclei.head()

,id,point
0,prg1_dk0,POINT (21.794 138.72)
1,prg1_dk1,POINT (19.615 141.887)
2,prg1_dk2,POINT (26.146 135.15)
3,prg1_dk3,POINT (21.147 141.275)
4,prg1_dk4,POINT (25.013 137.296)


In [7]:
nuclei.tail()

,id,point
4574,prg1_tz1215,POINT (53.12 8.437)
4575,prg1_tz1216,POINT (53.845 8.47)
4576,prg1_tz1217,POINT (52.603 8.754)
4577,prg1_tz1218,POINT (54.63 6.596)
4578,prg1_tz1219,POINT (51.939 9.409)


### Optional: Use a Small Sample

Uncomment this code cell to make a smaller data set for tests.

In [8]:
nuclei = nuclei.sample(250, random_state=0)


### Read Meitotic Stage Data

In [81]:
with pd.ExcelFile(MEIOTIC_STAGES, engine='calamine') as f:
    df = f.parse()

In [82]:
df['GonadID'] = df['GonadID'].str.split(".").str[:-1].str.join('.')

In [75]:
df = df.set_index('GonadID')

In [76]:
'210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched' in df.index

True

In [77]:
df.loc['210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched']

TZ_start              G
TZ_end                L
Pachy_end             Q
Notes               NaN
Stitched?          True
Rad-51 Modeled?    True
PRG-1 Modeled?     True
PGL-1 Modeled?      NaN
Exported Stats?    True
Name: 210924_DLW67_noAUX_HS_RAD51_AID_herm_g05_stitched, dtype: object

In [ ]:
def read_stages(fn):
    xls_file_name = fn.replace('.xlsx','.xls')
    with pd.ExcelFile(MEIOTIC_STAGES, engine='calamine') as f:
        df = f.parse().set_index('GonadID')
        row = df.loc[xls_file_name]
    return {s: row[s] for s in ['TZ_start','TZ_end','Pachy_end']}

**Optional:**  Uncomment this code cell to see the meiotic stages for the data set

In [ ]:
read_stages(TEST_DATA)

★ Read the meiotic stages, save them in global variables for future steps.

In [ ]:
stages = read_stages(TEST_DATA)

### Create Line Segments

Define a function that creates line segments from adjacent measurements.

##### Helper Function

In [ ]:
def orientation(sp, ep):
    if ep.x > sp.x:
        res = 'NE' if ep.y > sp.y else 'SE'
    else:
        res = 'NW' if ep.y > sp.y else 'SW'
    return res

In [ ]:
def create_segments(mf, sd=None):
    '''
    Make a frame where rows contain descriptions of line segments made by connecting
    adjacent points in the measurement frame.  Adds columns with attributes of each
    segment, including the parameters of the linear equation and segment length.

    Arguments:
        mf:  a data frame with names and locations of measurement points
        sd:  (optional) a dictionary with meiotic stage definitions

    Returns:
        a Pandas frame with line segments and their equations and lengths.
    '''
    df = gp.GeoDataFrame({
        'name': [mf.loc[i]['name'] + mf.loc[i+1]['name'] for i in range(len(mf)-1)],
        'head': [mf['point'].loc[i] for i in range(len(mf)-1)],
        'tail': [mf['point'].loc[i+1] for i in range(len(mf)-1)],
    }).set_geometry('head').set_geometry('tail')

    df['segment'] = [LineString([df['head'][i],df['tail'][i]]) for i in range(len(df))]
    df['orientation'] = [orientation(df['head'][i],df['tail'][i]) for i in range(len(df))]
    df['A'] = ((df['tail'].y - df['head'].y) / (df['tail'].x - df['head'].x))
    df['C'] = df['head'].y - df['head'].x*df['A']
    df['length'] = Point.distance(df['head'], df['tail'])
    df['pathlen'] = np.cumulative_sum(df['length'], include_initial=True)[:-1]

    if sd:
        df['stage'] = [('PMT' if p < sd['TZ_start'] else 'TZ' if p < sd['TZ_end'] else 'PACH') for p in mf['name'][0:-1]]
    else:
        df['stage'] = ['n/a'] * (len(mf)-1)
   
    return df.set_geometry('segment')

★ Save the line segments in a global variable for future steps.  Uncomment one of these lines, depending on whether or not the data set has meiotic stages.

In [ ]:
# segments = create_segments(measurements)
segments = create_segments(measurements, stages)

In [ ]:
segments

★ Save the total length of all line segments in a global variable

In [ ]:
germline_length = np.sum(segments['length'])

**Optional:**  Uncomment this code cell to see the first few line segments.

In [ ]:
segments.head()

In [ ]:
segments.tail()

**Optional:**  Uncomment the code in this cell to see a drawing of the line segments (using location data in the measurements frame)

In [ ]:
output_notebook()

p = figure(title='demo', x_axis_label='x', y_axis_label='y', match_aspect=True)
p.grid.grid_line_color = None

line = LineString(list(measurements.point))
dilated = line.buffer(10)
x, y = dilated.exterior.coords.xy
p.patch(list(x), list(y), fill_color='lightgray', line_color='lightgray')

xs = list(measurements['point'].x)
ys = list(measurements['point'].y)
ts = list(measurements['name'])
p.line(x=xs, y=ys)
p.scatter(x=xs, y=ys, size=5)
p.text(x=xs, y=ys, text=ts, x_offset=5, y_offset=10)
show(p)

**Optional:**  Uncomment the code in this cell to see a drawing of the nuclei with the line segments

In [ ]:
p = figure(title='germline', x_axis_label='x', y_axis_label='y', match_aspect=True)
p.grid.grid_line_color = None

line = LineString(list(measurements.point))
dilated = line.buffer(10)
x, y = dilated.exterior.coords.xy
p.patch(list(x), list(y), fill_color='lightgray', line_color='lightgray')

tf = nuclei[(nuclei.point.x > 22) | (nuclei.point.y < 100)]

yellow = Sunset10[5]
px = list(tf['point'].x)
py = list(tf['point'].y)
p.scatter(x=px, y=py, size=8, fill_color=yellow)

xs = list(measurements['point'].x)
ys = list(measurements['point'].y)
ts = list(measurements['name'])

p.line(x=xs, y=ys, color='black')
p.scatter(x=xs, y=ys, size=5, color='black')
p.text(x=xs, y=ys, text=ts, x_offset=5, y_offset=-5)

show(p)

### Make a Frame with All Combinations of Nuclei and Line Segments

Prepare for the step that computes distances from nuclei to segments by making a frame that has every combination of nuclei and segment descriptions.

★ Save the combined data in a global variable for future steps.

In [ ]:
combined = gp.GeoDataFrame.join(nuclei, segments, how="cross")

It's a lot of rows...

In [ ]:
len(combined)

Verify the combined frame has the expected number of rows.

In [ ]:
assert len(combined) == len(nuclei) * len(segments)

**Optional:**  Uncomment this code cell to see the first few lines in the combined frame.

In [ ]:
combined.head()

In [ ]:
combined.sample(n=10)

In [ ]:
type(combined)

In [ ]:
combined.dtypes

### Compute Distances

Define a function that computes three distances for each nucleus and line segment: the distance from a the nucleus to each end point (`pa` and `pb`), and the distance to the line segment (`pm`) 

In [ ]:
def add_distances(df: gp.GeoDataFrame):
    '''
    Create a new frame that summarizes the distances between nuclei and line segments.  The
    columns in the new frame will be:
    * `nuc_id`, the nucleus ID
    * `seg_name`, the name of the segment
    * `distance`, the distance from the nucleus to the segment
    * `location`, a string that identifies where the nucleus is closest (head, tail, middle of the segment)
    
    Arguments:
      df: a GeoDataFrame that has the line segments and their equations

    Returns:
      a frame with the distance values
    '''
    
    pos = gp.GeoDataFrame({
        'head': gp.GeoDataFrame.distance(df['point'],df['head']),
        'tail': gp.GeoDataFrame.distance(df['point'],df['tail']),
        'mid': gp.GeoDataFrame.distance(df['point'],df['segment']),
    })

    res = gp.GeoDataFrame({
        'nuc_id': df['id'],
        'seg_name': df['name'],
        'distance': pos.min(axis='columns'),
        'intersection': pos.idxmin(axis='columns')
    })

    return res

In [ ]:
distf = add_distances(combined)

In [ ]:
distf

### Groups

Group the distances by nucleus, and then find the shortest distance within each group.  The 
result of executing this expression is a column of row numbers, where each row number is the
row in the original frame that has the shortest distance to a nucleus.

In [ ]:
distf.groupby('nuc_id')[['distance']].min()

Use those column numbers to select the rows in the distance frame to get a complete description of a nucleus and the closest line segment.

In [ ]:
locs = distf.groupby('nuc_id')[['distance']].idxmin()
distf.loc[locs.distance]

Add the columns from the original frame so we have the data to compute intersections.

Save the result in a frame named `merged`.

In [ ]:
merged = gp.GeoDataFrame(distf.loc[locs.distance].join(combined[['point','head','tail','orientation','A','C','pathlen']]))

In [ ]:
merged.head()

In [ ]:
merged.dtypes

In [ ]:
type(merged)

### Compute Perpendicular Intersections

Add an intersection point to each row in the merged data.
* if the intersection is the middle of a line segment we need to compute the perpendicular intersection point using the equations of the line
* otherwise use one of the end points as the intersection location

In [ ]:
def add_intersections(df: gp.GeoDataFrame):
    '''
    Determine the location where nuclei intersect line segments.  

    Arguments:
      df: a frame that has line segments and their equations

    Returns:
      a copy of the frame with a new location column added
    '''
    mids = df[df['intersection']=='mid']
    xp = (mids.point.y - (-mids.point.x/mids.A) - mids.C) / (mids.A - (-1/mids.A))
    yp = (-xp/mids.A) + mids.point.y - (-mids.point.x/mids.A)
    mids['loc'] = [Point(xp.iloc[i],yp.iloc[i]) for i in range(len(mids))]

    heads = df[df['intersection']=='head']
    heads['loc'] = heads['head']

    tails = df[df['intersection']=='tail']
    tails['loc'] = tails['tail']

    res = pd.concat([mids,heads,tails]).set_geometry('loc')
    res['pathseg'] = gp.GeoDataFrame.distance(res['head'], res['loc'])
    res['pathloc'] = (res['pathlen'] + res['pathseg']) / germline_length
    
    return res

★ Save the intersections in a global variable for future steps.

In [ ]:
intersections = add_intersections(merged)

In [ ]:
intersections.head()

In [ ]:
intersections.tail()

### Assign a Meiotic Stage to Each Point

In [ ]:
def assign_stages(df):
    res = []
    for s in ['PMT','TZ']:
        sn = set(segments[segments['stage']==s].name)
        sf = intersections[intersections.seg_name.isin(sn)]
        sf['stage'] = s
        res.append(sf)
    sn = set(segments[segments['stage']=='PACH'].name)
    sf = intersections[intersections.seg_name.isin(sn)]
    delta = (sf['pathloc'].max() - sf['pathloc'].min()) / 3
    a = sf['pathloc'].min()
    for s in ['EP','MP','LP']:
        b = a + delta
        pf = intersections[(intersections.pathloc >= a) & (intersections.pathloc <= b)]
        pf['stage'] = s
        res.append(pf)
        a += delta
    return pd.concat(res)

In [ ]:
final = assign_stages(intersections)

In [ ]:
len(intersections), len(final)

In [ ]:
final.head()

In [ ]:
final.tail()

### Optional: Select a Random Subset to Display

Plotting the complete set of nuclei and their lines is too dense to make any sense. Execute this code cell to get a random sample for the plot.  

If you might want to reproduce the same sample later pass an optional start state parameter to `coords.sample`.

In [ ]:
# plot_coords = final.sample(len(intersections))    # <- choose all points
# plot_coords = final.sample(250, random_state=0)
# plot_coords = final[final['point'].y > 100].sample(50)
# plot_coords = final[(final['point'].y < 20) & (final['point'].x < 60)].sample(100)
plot_coords = final[(final.point.x > 22) | (final.point.y < 100)]

# see cell below that plots a histogram of distances; use this line to display points based on a distance cutoff
# plot_coords = intersections[intersections['distance'] < 20]
# plot_coords = intersections[intersections['distance'] < 12]

### Plot

Augment the plot shown earlier to include lines from nuclei to their closest line segment. 

In [ ]:
p = figure(title='demo', x_axis_label='x', y_axis_label='y', match_aspect=True)
p.grid.grid_line_color = None

for i, row in plot_coords.iterrows():
    xs = [row.point.x, row['loc'].x]
    ys = [row.point.y, row['loc'].y]
    p.line(xs, ys, line_dash='dashed')
    
yellow = Sunset10[5]
px = list(plot_coords['point'].x)
py = list(plot_coords['point'].y)
p.scatter(x=px, y=py, size=8, fill_color=yellow)


xs = list(measurements['point'].x)
ys = list(measurements['point'].y)
ts = list(measurements['name'])

p.line(x=xs, y=ys, color='black')
p.scatter(x=xs, y=ys, size=5, color='black')
p.text(x=xs, y=ys, text=ts, x_offset=5, y_offset=-5)

show(p)

In [ ]:
len(px)

In [ ]:
intersections['distance'].plot(kind='hist', bins=50)

In [ ]:
p = figure(title='demo', x_axis_label='x', y_axis_label='y', match_aspect=True)
p.grid.grid_line_color = None

df = pd.DataFrame({
    'x': plot_coords['point'].x,
    'y': plot_coords['point'].y,
    'c': plot_coords['pathloc'],
})

cmap = linear_cmap('c', palette="Viridis256", low=0, high=1)
r = p.scatter(x='x', y='y', color=cmap, size=8, source=df)

xs = list(measurements['point'].x)
ys = list(measurements['point'].y)
ts = list(measurements['name'])

p.line(x=xs, y=ys, color='black')
p.scatter(x=xs, y=ys, size=3, color='black')
p.text(x=xs, y=ys, text=ts, x_offset=5, y_offset=-5)

color_bar = r.construct_color_bar(padding=0, ticker=p.xaxis.ticker, formatter=p.xaxis.formatter)
p.add_layout(color_bar, 'below')

show(p)

In [ ]:
def vertical(row):
    match row.orientation:
        case 'SE': sign = 1 if row['point'].y > row['loc'].y else -1
        case 'SW': sign = 1 if row['point'].x > row['loc'].x else -1
        case 'NW': sign = 1 if row['point'].y < row['loc'].y else -1
        case 'NE': sign = 1 if row['point'].x < row['loc'].x else -1
    return sign * Point.distance(row['loc'],row['point'])

In [ ]:

p = figure(title='linearized', height=300, width=1000, x_axis_label='x', y_axis_label='y', y_range=(-50,30))
p.grid.grid_line_color = None

p.line(x=[-0.01,1.02], y=[0,0], color='lightgray', line_width=100)
p.line(x=[0,1], y=[0,0], color='black', line_width=2)

px = []
py = []

for i, row in plot_coords.iterrows():
    xs = [row.pathloc, row.pathloc]
    ys = [0, vertical(row)]
    px.append(xs[-1])
    py.append(ys[-1])
    p.line(xs, ys)

yellow = Sunset10[5]
p.scatter(x=px, y=py, size=8, fill_color=yellow)    


show(p)